# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all Record Sets in the dataset
print('Record Sets:')
record_sets = []
for rs in metadata.record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', '')}, description: {rs.get('description', '')}")
    record_sets.append(rs['@id'])
    # Print fields for each record set
    print('  Fields:')
    for field in rs.get('fields', []):
        print(f"    - @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this notebook, we'll choose the first record set for demonstration
if len(record_sets)==0:
    raise Exception('No record sets found in the metadata.')
example_record_set_id = record_sets[0] # e.g., 'cr:RecordSet1'

# Load all records from the chosen record set into a pandas DataFrame
records = list(dataset.records(record_set=example_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records from RecordSet @id={example_record_set_id}")
print("Available columns (@id):", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
You can adapt this cell to suit the structure of your dataset and reference fields by their `@id`.

In [ ]:
# Display numeric fields by their @id
numeric_types = ["Float", "Integer", "Number", "schema:Float", "schema:Integer", "schema:Number"]
numeric_candidate_ids = []
for rs in metadata.record_sets:
    if rs['@id'] == example_record_set_id:
        for field in rs.get('fields', []):
            # Check if field dataType matches one of the numeric types
            if field.get('dataType', '') in numeric_types:
                numeric_candidate_ids.append(field['@id'])

print("Numeric fields (by @id) in this record set:", numeric_candidate_ids)

# Select a numeric field (first found)
if not numeric_candidate_ids:
    raise Exception('No numeric fields found!')
numeric_field_id = numeric_candidate_ids[0]

# Threshold for filtering
threshold = df[numeric_field_id].quantile(0.75) # for example: filter for the top 25%
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} rows)")
print(filtered_df.head())

# Normalize the chosen numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field
cat_candidate_ids = []
for rs in metadata.record_sets:
    if rs['@id'] == example_record_set_id:
        for field in rs.get('fields', []):
            if field.get('dataType', '') not in numeric_types:
                if field['@id'] in df.columns and df[field['@id']].dtype == 'object':
                    cat_candidate_ids.append(field['@id'])
group_field = cat_candidate_ids[0] if cat_candidate_ids else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print('No suitable categorical field for grouping found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If a group field exists, plot grouped means
if group_field:
    plt.figure(figsize=(10,6))
    mean_values = df.groupby(group_field)[numeric_field_id].mean().sort_values()
    mean_values.plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.